# Copyright (c) 2026 Khush Patil. All Rights Reserved.
# This code is for personal/educational viewing only. 
# Unauthorized copying, modification, or distribution is strictly prohibited.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
import timm
import os

# --- 1. Settings & Toggle ---
# CHANGE THIS TO 'MNIST' OR 'EMNIST'
DATASET_TO_TRAIN = 'EMNIST'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# EMNIST images are famously rotated 90 degrees and flipped by default.
# This custom transform fixes them so the models learn the right orientation.
def emnist_fix(img):
    return TF.hflip(TF.rotate(img, -90))

if DATASET_TO_TRAIN == 'EMNIST':
    base_transforms = [transforms.Lambda(emnist_fix)]
else:
    base_transforms = []

# Standardizing to 224x224 and 3 channels so our Devanagari architectures work flawlessly
transform = transforms.Compose(base_transforms + [
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 2. Auto-Download Datasets ---
print(f"Downloading and loading {DATASET_TO_TRAIN}...")

if DATASET_TO_TRAIN == 'MNIST':
    train_data = datasets.MNIST(root='/content/data', train=True, download=True, transform=transform)
    test_data = datasets.MNIST(root='/content/data', train=False, download=True, transform=transform)
elif DATASET_TO_TRAIN == 'EMNIST':
    # We use the 'balanced' split (47 classes of letters and digits) for a solid benchmark challenge
    train_data = datasets.EMNIST(root='/content/data', split='balanced', train=True, download=True, transform=transform)
    test_data = datasets.EMNIST(root='/content/data', split='balanced', train=False, download=True, transform=transform)

# Using a slightly larger batch size to speed up the T4 GPU
train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=128, shuffle=False)

num_classes = len(train_data.classes)
print(f"✅ Dataset loaded! Number of classes: {num_classes}")

# --- 3. Define the Models (Exact same architecture as Devanagari) ---
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

vit_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=num_classes)

class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        resnet = timm.create_model('resnet18', pretrained=True)
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layer, num_layers=2)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn_backbone(x)
        x = x.flatten(2).permute(0, 2, 1)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

# --- 4. Training & Evaluation Setup ---
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def train_model(model, name, epochs=3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print(f"\n--- Training {name} on {DATASET_TO_TRAIN} ---")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        test_acc = evaluate_model(model, test_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Test Accuracy: {test_acc:.2f}%")

    # Save the model securely to a new folder in your Drive
    save_dir = f'/content/drive/MyDrive/Devanagari_Project/benchmark_models/{DATASET_TO_TRAIN.lower()}'
    os.makedirs(save_dir, exist_ok=True)
    save_path = f'{save_dir}/{name}.pth'
    torch.save(model.state_dict(), save_path)
    print(f"✅ Saved {name} to: {save_path}")

# --- 5. Execute Execution ---
cnn_model = SimpleCNN()
hybrid_model = HybridModel()

# We can use fewer epochs here because MNIST/EMNIST are much simpler than Devanagari
train_model(cnn_model, f"cnn_{DATASET_TO_TRAIN.lower()}", epochs=3)
train_model(vit_model, f"vit_{DATASET_TO_TRAIN.lower()}", epochs=2)
train_model(hybrid_model, f"hybrid_{DATASET_TO_TRAIN.lower()}", epochs=3)

✅ Using device: cuda


100%|██████████| 562M/562M [00:02<00:00, 262MB/s]


✅ Dataset loaded! Number of classes: 47


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]


--- Training cnn_emnist on EMNIST ---
Epoch 1/3 | Loss: 1.1313 | Test Accuracy: 79.85%
Epoch 2/3 | Loss: 0.5131 | Test Accuracy: 82.22%
Epoch 3/3 | Loss: 0.3992 | Test Accuracy: 82.13%
✅ Saved cnn_emnist to: /content/drive/MyDrive/Devanagari_Project/benchmark_models/emnist/cnn_emnist.pth

--- Training vit_emnist on EMNIST ---
Epoch 1/2 | Loss: 2.7581 | Test Accuracy: 73.21%
Epoch 2/2 | Loss: 0.5564 | Test Accuracy: 82.15%
✅ Saved vit_emnist to: /content/drive/MyDrive/Devanagari_Project/benchmark_models/emnist/vit_emnist.pth

--- Training hybrid_emnist on EMNIST ---
Epoch 1/3 | Loss: 0.5049 | Test Accuracy: 87.06%
Epoch 2/3 | Loss: 0.3235 | Test Accuracy: 87.00%
Epoch 3/3 | Loss: 0.2903 | Test Accuracy: 88.63%
✅ Saved hybrid_emnist to: /content/drive/MyDrive/Devanagari_Project/benchmark_models/emnist/hybrid_emnist.pth
